# 📊 Sales Prediction — End-to-End ML Project

Predict future sales from advertising spend, platform, customer segment, campaign performance, and seasonal trends.

**Models:** Linear Regression · Decision Tree · Random Forest · Gradient Boosting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2563EB','#DC2626','#16A34A','#D97706','#7C3AED','#DB2777']
sns.set_palette(PALETTE)
print('Libraries loaded!')

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv('../data/sales_data.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'Duplicates: {df.duplicated().sum()}')

## 2. Data Preprocessing

In [ ]:
# Drop duplicates
df = df.drop_duplicates()

# Fill missing numerics with median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# IQR outlier clipping
for col in ['Ad_Spend','Sales']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

# Encode categoricals
cat_cols = ['Platform','Customer_Segment','Campaign_Type','Region']
for col in cat_cols:
    le = LabelEncoder()
    df[col+'_Enc'] = le.fit_transform(df[col].astype(str))

# Feature engineering
df['Log_Ad_Spend']      = np.log1p(df['Ad_Spend'])
df['Log_Traffic']       = np.log1p(df['Website_Traffic'])
df['Spend_per_Channel'] = df['Ad_Spend'] / df['Channels_Used'].replace(0,1)
df['Score_x_Season']    = df['Campaign_Score'] * df['Seasonal_Factor']
df['Quarter']           = df['Date'].dt.quarter
print(f'Processed shape: {df.shape}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df['Sales'], bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].set_title('Raw Sales Distribution')
axes[1].hist(np.log1p(df['Sales']), bins=50, color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[1].set_title('Log-Transformed Sales')
plt.suptitle('Sales Distribution', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12,5))
monthly = df.groupby('Month')['Sales'].mean()
plt.plot(monthly.index, monthly.values, marker='o', linewidth=2.5, color=PALETTE[0])
plt.fill_between(monthly.index, monthly.values, alpha=0.15, color=PALETTE[0])
plt.xticks(range(1,13),['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.title('Monthly Seasonal Sales Trend', fontweight='bold')
plt.xlabel('Month'); plt.ylabel('Avg Sales (USD)'); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(15,5))
sns.boxplot(data=df, x='Platform', y='Sales', palette=PALETTE, ax=axes[0])
axes[0].set_title('Sales by Platform'); axes[0].tick_params(axis='x',rotation=30)
sns.boxplot(data=df, x='Customer_Segment', y='Sales', palette=PALETTE, ax=axes[1])
axes[1].set_title('Sales by Customer Segment'); axes[1].tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
num_feats = ['Ad_Spend','Campaign_Score','Competitor_Index','Economic_Index',
             'Discount_Pct','Channels_Used','CSAT_Score','Seasonal_Factor','Sales']
corr = df[num_feats].corr()
plt.figure(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Model Development

In [ ]:
features = [
    'Log_Ad_Spend','Campaign_Score','Competitor_Index','Economic_Index',
    'Discount_Pct','Channels_Used','CSAT_Score','Log_Traffic',
    'Seasonal_Factor','Spend_per_Channel','Score_x_Season',
    'Month','Quarter',
    'Platform_Enc','Customer_Segment_Enc','Campaign_Type_Enc','Region_Enc'
]
X = df[features].fillna(df[features].median())
y = df['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
models = {
    'Linear Regression' : LinearRegression(),
    'Decision Tree'     : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    Xtr = X_train_sc if name == 'Linear Regression' else X_train
    Xte = X_test_sc  if name == 'Linear Regression' else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R2':r2,'MAE':mae,'RMSE':rmse,'pred':y_pred}
    print(f'[{name}]  R2={r2:.4f}  MAE={mae:,.0f}  RMSE={rmse:,.0f}')

## 5. Model Evaluation

In [ ]:
metrics_df = pd.DataFrame({k:{m:v for m,v in results[k].items() if m!='pred'}
                           for k in results}).T.round(4)
metrics_df

In [ ]:
best_name = max(results, key=lambda k: results[k]['R2'])
y_pred_best = results[best_name]['pred']

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle(f'{best_name} — Actual vs Predicted', fontweight='bold')
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color=PALETTE[0], s=20)
lims = [min(y_test.min(),y_pred_best.min()), max(y_test.max(),y_pred_best.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
residuals = y_test.values - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color=PALETTE[2], s=20)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residuals')
plt.tight_layout(); plt.show()
print(f'Best: {best_name}  R2={results[best_name]["R2"]:.4f}')

## 6. Feature Importance & Sales Forecast

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=features).sort_values()
plt.figure(figsize=(10,7))
fi.plot(kind='barh', color=PALETTE[0])
plt.title('Random Forest Feature Importances', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
best_model = models[best_name]
spend_vals = [5000,10000,20000,40000,80000,150000]
preds = []
base = {'Log_Ad_Spend':np.log1p(20000),'Campaign_Score':75,'Competitor_Index':0.5,
        'Economic_Index':1.0,'Discount_Pct':10,'Channels_Used':3,'CSAT_Score':4.0,
        'Log_Traffic':np.log1p(8000),'Seasonal_Factor':1.0,'Spend_per_Channel':20000/3,
        'Score_x_Season':75.0,'Month':6,'Quarter':2,
        'Platform_Enc':4,'Customer_Segment_Enc':3,'Campaign_Type_Enc':3,'Region_Enc':0}
for s in spend_vals:
    row = {**base,'Log_Ad_Spend':np.log1p(s),'Spend_per_Channel':s/3}
    preds.append(best_model.predict(pd.DataFrame([row])[features])[0])

plt.figure(figsize=(11,5))
labels = [f'${s//1000}K' for s in spend_vals]
plt.plot(labels, preds, marker='o', linewidth=2.5, color=PALETTE[0], markersize=9)
plt.fill_between(range(len(labels)), preds, alpha=0.12, color=PALETTE[0])
plt.title('Sales Forecast vs Advertising Spend', fontweight='bold')
plt.xlabel('Ad Spend'); plt.ylabel('Predicted Sales')
plt.tight_layout(); plt.show()
for s,p in zip(spend_vals,preds):
    print(f'  Ad Spend ${s:>8,.0f}  →  Predicted Sales ${p:>10,.2f}')

## 7. Save Best Model

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler,     '../models/scaler.pkl')
print(f'Saved: {best_name}')